# ReClip - Colab Wav2Lip API

Bu notebook GPU uzerinde **Wav2Lip** calistirir ve ReClip'in cagirabilecegi bir HTTP endpoint acar.

### Adimlar
1. **Runtime > Change runtime type > T4 GPU** sec
2. Hucreleri sirayla calistir
3. Son hucredeki `LIPSYNC URL` degerini ReClip Ayarlar > Lipsync URL alanina yapistir

### Ngrok
- [ngrok.com](https://ngrok.com) -> Dashboard > Your Authtoken
- `NGROK_TOKEN` degiskenine yapistir

In [ ]:
NGROK_TOKEN = ""  # ngrok.com'dan al

In [ ]:
# Wav2Lip repo + bagimliliklar (Colab Py3.12 uyumlu)
!git clone https://github.com/Rudrabha/Wav2Lip /content/Wav2Lip 2>/dev/null || echo already cloned
%cd /content/Wav2Lip
!pip install -q librosa opencv-python-headless fastapi uvicorn python-multipart pyngrok

# librosa 0.10+ keyword-only API icin audio.py yamasi
import re, pathlib
p = pathlib.Path("/content/Wav2Lip/audio.py")
s = p.read_text()
s = re.sub(r"librosa\.filters\.mel\(hp\.sample_rate,\s*hp\.n_fft,\s*n_mels=hp\.num_mels",
           "librosa.filters.mel(sr=hp.sample_rate, n_fft=hp.n_fft, n_mels=hp.num_mels", s)
s = s.replace("librosa.core.stft", "librosa.stft")
s = re.sub(r"librosa\.filters\.mel\(([^,)]+),\s*([^,)]+),\s*n_mels=", r"librosa.filters.mel(sr=, n_fft=, n_mels=", s)
p.write_text(s)
print("audio.py patched")

In [ ]:
# Checkpoint indir (HF mirror - orijinal gdrive cogu zaman 404)
import os
os.makedirs('/content/Wav2Lip/checkpoints', exist_ok=True)
!wget -q -O /content/Wav2Lip/checkpoints/wav2lip_gan.pth https://huggingface.co/camenduru/Wav2Lip/resolve/main/wav2lip_gan.pth
# s3fd yuz dedektoru
!mkdir -p /content/Wav2Lip/face_detection/detection/sfd
!wget -q -O /content/Wav2Lip/face_detection/detection/sfd/s3fd.pth https://www.adrianbulat.com/downloads/python-fan/s3fd-619a316812.pth || \
    wget -q -O /content/Wav2Lip/face_detection/detection/sfd/s3fd.pth https://huggingface.co/camenduru/Wav2Lip/resolve/main/s3fd.pth
print('checkpoint size:', os.path.getsize('/content/Wav2Lip/checkpoints/wav2lip_gan.pth'))
print('s3fd size:', os.path.getsize('/content/Wav2Lip/face_detection/detection/sfd/s3fd.pth'))

In [ ]:
import os, tempfile, threading, subprocess, uuid, shutil
import uvicorn
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import FileResponse, JSONResponse
from pyngrok import ngrok

WAV2LIP_DIR = '/content/Wav2Lip'
CKPT = f'{WAV2LIP_DIR}/checkpoints/wav2lip_gan.pth'
WORK = '/content/lipsync_work'
os.makedirs(WORK, exist_ok=True)

api = FastAPI(title='ReClip Lip-sync API')

@api.get('/health')
def health():
    return {'status': 'ok', 'model': 'wav2lip_gan'}

@api.post('/lipsync')
async def lipsync(video: UploadFile = File(...), audio: UploadFile = File(...)):
    jid = uuid.uuid4().hex[:8]
    vpath = f'{WORK}/{jid}_in.mp4'
    apath = f'{WORK}/{jid}_in.wav'
    opath = f'{WORK}/{jid}_out.mp4'
    with open(vpath, 'wb') as f:
        f.write(await video.read())
    with open(apath, 'wb') as f:
        f.write(await audio.read())
    try:
        r = subprocess.run([
            'python', f'{WAV2LIP_DIR}/inference.py',
            '--checkpoint_path', CKPT,
            '--face', vpath,
            '--audio', apath,
            '--outfile', opath,
            '--pads', '0', '10', '0', '0',
            '--resize_factor', '1',
            '--nosmooth',
        ], cwd=WAV2LIP_DIR, capture_output=True, text=True)
        if r.returncode != 0 or not os.path.exists(opath):
            return JSONResponse({'error': r.stderr[-2000:] or r.stdout[-2000:]}, status_code=500)
        return FileResponse(opath, media_type='video/mp4', filename=f'{jid}_lipsync.mp4')
    finally:
        for p in (vpath, apath):
            try: os.unlink(p)
            except OSError: pass

if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)
tunnel = ngrok.connect(8000, bind_tls=True)
print('\n' + '=' * 55)
print(f'  LIPSYNC URL: {tunnel.public_url}')
print('=' * 55)
print('Bu URL\'yi ReClip > Ayarlar > Lipsync URL alanina gir.')
print('=' * 55 + '\n')

def _run():
    uvicorn.run(api, host='0.0.0.0', port=8000, log_level='warning')
threading.Thread(target=_run, daemon=True).start()
print('Sunucu calisiyor. Notebook acik kaldigi surece aktif.')